# Setup, full data audit, and reference splits

Disk-backed DuckDB checks cover all six files, complete truth coverage, unique IDs, external ownership and country compatibility. Deterministic splits are stratified by country and match count; singleton references are retained. Inputs and outputs are fingerprinted. Shared ownership fails pending connected-component splitting. Inspect split counts: tiny strata can leave splits empty.

The complete audit is opt-in. Database size is checked on reuse; exported truth/split files also have SHA-256 checksums.

In [ ]:
from pathlib import Path
import csv
import hashlib
import importlib.metadata
import json
import math
import os
import platform
import time
import uuid

SOURCE_COLUMNS = ["entity_id", "business_name", "business_address", "country"]
SPLIT_NAMES = ("fit", "early_stop", "threshold_tune", "holdout")

def stable_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,
                                    allow_nan=False).encode("utf-8")).hexdigest()

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False),
                         encoding="utf-8")
    temporary.replace(path)

def load_helpers(path, expected_hash=None):
    """Execute trusted project definition cells only, never notebook run cells."""
    path = Path(path).resolve()
    if path.parent.name != "anshp" or path.suffix != ".ipynb":
        raise ValueError("Expected a local anshp notebook")
    nb = json.loads(path.read_text(encoding="utf-8"))
    sources = ["".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code"
               and "er-definitions" in c.get("metadata", {}).get("tags", [])]
    digest = stable_hash(sources)
    if not sources or (expected_hash is not None and digest != expected_hash):
        raise ValueError("Missing definitions or helper contract mismatch")
    namespace = {"__file__": str(path), "__name__": "notebook_helpers"}
    for i, source in enumerate(sources):
        exec(compile(source, f"{path.name}:definitions:{i}", "exec"), namespace)
    return namespace, digest

def sql_literal(value):
    return "'" + str(value).replace("'", "''") + "'"

def connect_database(path=":memory:", read_only=False):
    import duckdb
    con = duckdb.connect(str(path), read_only=read_only)
    con.execute("SET memory_limit='4GB'")
    con.execute("SET threads=4")
    con.execute("SET preserve_insertion_order=false")
    return con

def require_zero(con, sql, message):
    rows = con.execute(sql).fetchmany(5)
    if rows:
        raise ValueError(f"{message}: {rows}")

def file_entry(path, root, checksum=True):
    path, root = Path(path), Path(root)
    result = {"path": str(path.relative_to(root)), "bytes": path.stat().st_size}
    if checksum:
        result["sha256"] = sha256_file(path)
    return result

def require_manifest(path, stage, verify_hashes=True):
    path = Path(path)
    m = json.loads(path.read_text(encoding="utf-8"))
    if m.get("stage") != stage or not m.get("complete"):
        raise ValueError(f"Incomplete or incompatible {stage} manifest")
    for item in m["files"]:
        f = path.parent / item["path"]
        if not f.is_file() or f.stat().st_size != item["bytes"]:
            raise ValueError(f"Missing or changed artifact: {f}")
        if verify_hashes and "sha256" in item and sha256_file(f) != item["sha256"]:
            raise ValueError(f"Checksum mismatch: {f}")
    return m

def environment_snapshot(include_gpu=False):
    result = {"python": platform.python_version(), "packages": {}}
    for package in ("duckdb", "pyarrow", "ftfy", "anyascii", "torch", "transformers",
                    "peft", "accelerate"):
        try:
            result["packages"][package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            result["packages"][package] = None
    if include_gpu:
        import torch
        result["cuda_available"] = torch.cuda.is_available()
        result["gpus"] = [{"name": torch.cuda.get_device_properties(i).name,
                           "bytes": torch.cuda.get_device_properties(i).total_memory}
                          for i in range(torch.cuda.device_count())]
    return result

def build_data_audit(data_dir, output_dir, config, split_overrides=None):
    """Disk-backed source/truth audit. Publication happens only after all checks pass."""
    data_dir, out = Path(data_dir), Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    inputs = []
    for dataset in ("train", "test"):
        for source in (1, 2, 3):
            p = data_dir / dataset / f"{dataset}_source{source}.tsv"
            with p.open(encoding="utf-8-sig", newline="") as f:
                if next(csv.reader(f, delimiter="\t")) != SOURCE_COLUMNS:
                    raise ValueError(f"Unexpected source header: {p}")
            inputs.append({"dataset": dataset, "source": source, "path": str(p.resolve()),
                           "sha256": sha256_file(p)})
    truth_path = data_dir / "train" / "train_ground_truth.tsv"
    with truth_path.open(encoding="utf-8-sig", newline="") as f:
        truth_columns = next(csv.reader(f, delimiter="\t"))
    if truth_columns != ["source1_entity_id", "matched_entity_ids"]:
        raise ValueError(f"Unexpected truth header: {truth_columns}")
    fractions = config.get("split_fractions", dict(zip(SPLIT_NAMES, (.7, .1, .1, .1))))
    if set(fractions) != set(SPLIT_NAMES) or any(v <= 0 for v in fractions.values()) \
            or not math.isclose(sum(fractions.values()), 1.0):
        raise ValueError("Split fractions must be positive and sum to one")
    identity = {"version": "audit-v3", "inputs": inputs,
                "helpers": load_helpers(Path(__file__) if "__file__" in globals() else next(
                    p/"anshp/00_setup_and_data_audit.ipynb" for p in (Path.cwd(), *Path.cwd().parents)
                    if (p/"anshp/config.json").is_file()))[1],
                "truth_sha256": sha256_file(truth_path), "seed": config["seed"],
                "fractions": fractions, "overrides": split_overrides}
    fingerprint = stable_hash(identity)
    manifest_path = out / "manifest.json"
    if manifest_path.exists():
        cached = require_manifest(manifest_path, "audit")
        if cached["fingerprint"] == fingerprint:
            return cached
    started = time.perf_counter()
    db = out / f"audit-{fingerprint[:16]}.duckdb"
    temp = out / f"audit-{uuid.uuid4().hex}.tmp.duckdb"
    con = connect_database(temp)
    files = []
    try:
        con.execute("""CREATE TABLE records(dataset VARCHAR, source INTEGER,
                       entity_id VARCHAR, business_name VARCHAR, business_address VARCHAR,
                       country VARCHAR)""")
        source_schema = "{" + ",".join(sql_literal(k) + ":'VARCHAR'" for k in SOURCE_COLUMNS) + "}"
        for item in inputs:
            con.execute(f"""INSERT INTO records SELECT {sql_literal(item['dataset'])},
                {item['source']}, entity_id, coalesce(business_name,''),
                coalesce(business_address,''), trim(country)
                FROM read_csv({sql_literal(item['path'])}, delim='\t', header=true,
                columns={source_schema}, nullstr='', strict_mode=true)""")
        require_zero(con, """SELECT dataset, entity_id FROM records GROUP BY ALL
                             HAVING count(*)>1""", "Duplicate source ID")
        require_zero(con, """SELECT entity_id FROM records WHERE entity_id IS NULL OR
            entity_id='' OR entity_id!=trim(entity_id) OR regexp_matches(entity_id, '[,\t\r\n]')
            OR country IS NULL OR country=''""", "Invalid ID or country")
        con.execute(f"""CREATE TABLE truth_rows AS SELECT source1_entity_id reference_id,
            coalesce(matched_entity_ids,'') ids FROM read_csv({sql_literal(truth_path)},
            delim='\t',header=true,columns={{'source1_entity_id':'VARCHAR',
            'matched_entity_ids':'VARCHAR'}},nullstr='',strict_mode=true)""")
        require_zero(con, """SELECT reference_id FROM truth_rows GROUP BY 1 HAVING
                            count(*)>1 OR reference_id IS NULL""", "Duplicate/missing truth reference")
        require_zero(con, """SELECT coalesce(r.entity_id,t.reference_id) FROM
            (SELECT * FROM records WHERE dataset='train' AND source=1) r
            FULL JOIN truth_rows t ON r.entity_id=t.reference_id
            WHERE r.entity_id IS NULL OR t.reference_id IS NULL""", "Truth coverage mismatch")
        con.execute("""CREATE TABLE truth AS SELECT reference_id, trim(x) candidate_id
            FROM truth_rows, unnest(string_split(ids,',')) u(x) WHERE ids!=''""")
        require_zero(con, """SELECT reference_id,candidate_id FROM truth GROUP BY ALL
                            HAVING count(*)>1""", "Repeated truth pair")
        require_zero(con, """SELECT t.* FROM truth t LEFT JOIN records r ON
            r.dataset='train' AND r.source IN (2,3) AND r.entity_id=t.candidate_id
            WHERE r.entity_id IS NULL""", "Unknown external truth ID")
        require_zero(con, """SELECT candidate_id FROM truth GROUP BY 1 HAVING
            count(DISTINCT reference_id)>1""", "Shared external ownership; group connected references first")
        require_zero(con, """SELECT t.* FROM truth t JOIN records q ON q.dataset='train'
            AND q.entity_id=t.reference_id JOIN records d ON d.dataset='train'
            AND d.entity_id=t.candidate_id WHERE q.country!=d.country""",
            "Country partition would discard true links")
        con.execute("""CREATE TABLE reference_stats AS SELECT r.entity_id reference_id,
            r.country, count(t.candidate_id)::INTEGER gold_count FROM records r
            LEFT JOIN truth t ON t.reference_id=r.entity_id WHERE r.dataset='train'
            AND r.source=1 GROUP BY 1,2""")
        if split_overrides is not None:
            con.execute("CREATE TABLE assignments(reference_id VARCHAR, split VARCHAR)")
            con.executemany("INSERT INTO assignments VALUES (?,?)", list(split_overrides.items()))
            require_zero(con, """SELECT coalesce(r.reference_id,a.reference_id)
                FROM reference_stats r FULL JOIN assignments a USING(reference_id)
                WHERE r.reference_id IS NULL OR a.reference_id IS NULL""", "Split coverage mismatch")
            if not set(split_overrides.values()).issubset(SPLIT_NAMES):
                raise ValueError("Invalid split name")
            con.execute("""CREATE TABLE splits AS SELECT r.*,a.split FROM reference_stats r
                           JOIN assignments a USING(reference_id)""")
        else:
            cutoffs, total = [], 0
            for name in SPLIT_NAMES[:-1]:
                total += fractions[name]
                cutoffs.append(f"WHEN (rn-0.5)/n < {total} THEN {sql_literal(name)}")
            con.execute(f"""CREATE TABLE splits AS SELECT reference_id,country,gold_count,
                CASE {' '.join(cutoffs)} ELSE 'holdout' END split FROM
                (SELECT *,row_number() OVER w rn,
                count(*) OVER(PARTITION BY country,least(gold_count,6)) n
                FROM reference_stats WINDOW w AS (PARTITION BY country,least(gold_count,6)
                ORDER BY md5({sql_literal(str(config['seed']) + ':')}||reference_id),reference_id))""")
        con.execute("""CREATE TABLE ownership AS SELECT candidate_id,reference_id owner_reference_id,
                       split owner_split FROM truth JOIN splits USING(reference_id)""")
        counts = con.execute("""SELECT dataset,source,country,count(*) n,
            sum(business_name='') blank_name,sum(business_address='') blank_address
            FROM records GROUP BY ALL ORDER BY 1,2,3""").fetchall()
        split_counts = con.execute("""SELECT split,country,count(*),sum(gold_count=0),
            sum(gold_count) FROM splits GROUP BY ALL ORDER BY 1,2""").fetchall()
        summary = con.execute("""SELECT count(*),sum(gold_count=0),sum(gold_count),
                                 max(gold_count) FROM splits""").fetchone()
        for table in ("splits", "ownership", "truth"):
            p = out / f"{table}-{fingerprint[:16]}.parquet"
            tmp = p.with_suffix(".tmp.parquet")
            con.execute(f"COPY {table} TO {sql_literal(tmp)} (FORMAT PARQUET,COMPRESSION ZSTD)")
            tmp.replace(p)
            files.append(file_entry(p, out))
        con.execute("CHECKPOINT")
        con.close()
        con = None
        temp.replace(db)
        files.append(file_entry(db, out, checksum=False))
        manifest = {"stage": "audit", "complete": True, "fingerprint": fingerprint,
                    "identity": identity, "database": db.name, "files": files,
                    "source_counts": counts, "split_counts": split_counts,
                    "truth_summary": summary, "country_partition_verified": True,
                    "environment": environment_snapshot(),
                    "seconds": time.perf_counter()-started}
        atomic_json(manifest_path, manifest)
        return manifest
    finally:
        if con is not None:
            con.close()


In [ ]:
from pathlib import Path
import json
PROJECT_DIR = next(p / "anshp" for p in (Path.cwd(), *Path.cwd().parents)
                   if (p / "anshp" / "config.json").is_file())
DATA_DIR = PROJECT_DIR.parent / "6ab10eb3b23ba_student_resource" / "student_resource" / "dataset"
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
CONFIG = json.loads((PROJECT_DIR / "config.json").read_text(encoding="utf-8"))

print(environment_snapshot())

In [ ]:
RUN_AUDIT = False
if RUN_AUDIT:
    audit = build_data_audit(DATA_DIR, ARTIFACT_DIR / 'audit', CONFIG)
    print(audit['truth_summary'], audit['split_counts'])
